# MTech Bug-Fix Agent: Colab Feasibility Check

This notebook checks the Colab runtime, verifies one small Python bug benchmark record, and checks the Qwen model. The official SWE-bench test harness is too large for the current free Colab runtime, so we are checking Goldset as a smaller alternative. Run cells from top to bottom, but you can skip the model download cell if you only want the benchmark check.

In Colab, choose **Runtime → Change runtime type** and select a GPU if available.

In [ ]:
import platform
import sys
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('GPU memory (GiB):', round(props.total_memory / (1024 ** 3), 2))
else:
    print('GPU: none selected or currently available')
print('Python executable:', sys.executable)

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GiB): 14.56
Python executable: /usr/bin/python3


## Download and install the project

The repository is public. This cell clones it into Colab's temporary storage and installs the lightweight project package. Colab storage resets when its runtime is discarded, so rerun this notebook after a reset.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/pisalaarti05/mtech-bug-fix-agent.git'
PROJECT_DIR = Path('/content/mtech-bug-fix-agent')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--editable', str(PROJECT_DIR)], check=True)
print('Project installed from:', PROJECT_DIR)

Project installed from: /content/mtech-bug-fix-agent


## Confirm the command is ready

This displays the program's options. A successful result means Colab can import the project. Do not choose a model yet; first share the runtime details above.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'bugfix_agent.cli', '--help'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'bugfix_agent.cli', '--help'], returncode=0)

## Check one SWE-bench Lite task and Colab capacity

SWE-bench Lite is a suitable issue-driven candidate, but its official evaluation runs Docker containers and needs at least 120 GB of free disk. This cell only reads one task record and checks your current Colab resources; it does not reveal the reference patch or run benchmark tests. If Docker or disk is insufficient, do not claim official SWE-bench scores from this runtime.

In [ ]:
import shutil
import subprocess
import sys

# Read one benchmark example without downloading the entire dataset.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3.0'], check=True)
from datasets import load_dataset

tasks = load_dataset('princeton-nlp/SWE-bench_Lite', split='test', streaming=True)
task = next(iter(tasks))
print('Instance:', task['instance_id'])
print('Repository:', task['repo'])
print('Base commit:', task['base_commit'])
print('Issue description (first 1800 chars):')
print(task['problem_statement'][:1800])

disk = shutil.disk_usage('/content')
print(f'Free Colab disk: {disk.free / (1024**3):.1f} GiB')
docker = subprocess.run(['bash', '-lc', 'docker --version'], capture_output=True, text=True)
print('Docker available:', docker.returncode == 0)
if docker.returncode == 0:
    print(docker.stdout.strip())
else:
    print('Docker is not available in this runtime.')
print('Official SWE-bench harness minimum free disk: 120 GB.')


Instance: astropy__astropy-12907
Repository: astropy/astropy
Base commit: d16bfe05a744909de4b27f5875fe0d4ed41ce607
Issue description (first 1800 chars):
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however,

## Verify one record from the smaller Goldset benchmark

Goldset provides real Python bug records, project commit IDs, license metadata, and a regression test. This cell picks one file-level record from after 2025-01-01, prints its identifying details, then asks Goldset's own verifier to check that the recorded test fails before the fix and passes after it. It does not show the reference fix to the model. This is a feasibility check, not an agent score.

The verifier clones a public repository and runs its test code in this Colab session. It does not use Docker. A record may be reported as skipped if its dependencies or tests cannot run in the available environment.

In [29]:
from pathlib import Path
import json
import subprocess
import sys

GOLDSET_DIR = Path('/content/goldset')
if not GOLDSET_DIR.exists():
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', 'https://github.com/andysalvo/goldset.git', str(GOLDSET_DIR)], check=True)

# Install Goldset's verifier and uv, which can run projects with an older Python.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(GOLDSET_DIR), 'uv'], check=True)

records = [json.loads(line) for line in (GOLDSET_DIR / 'oracles.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
candidates = [r for r in records if r.get('granularity') == 'file' and r.get('committed_at', '') >= '2025-01-01']
if not candidates:
    raise RuntimeError('No file-level Goldset records dated 2025-01-01 or later were found.')
record = sorted(candidates, key=lambda r: (r['committed_at'], r['repo'], r['fingerprint']))[0]
goldset_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=GOLDSET_DIR, check=True, capture_output=True, text=True).stdout.strip()
print('Goldset repository revision:', goldset_sha)
print('Record:', record['fingerprint'])
print('Fix date:', record['committed_at'])
print('Project:', record['repo'])
print('Project license:', record['license'])
print('Bug specification:', record['spec'][:1200])
print('Running one-record verification now...')
verification = subprocess.run([sys.executable, 'verify.py', '--corpus', 'oracles.jsonl', '--fingerprint', record['fingerprint']], cwd=GOLDSET_DIR, capture_output=True, text=True)
print(verification.stdout)
if verification.stderr.strip():
    print('Verifier messages:', verification.stderr[-3000:])
print('Verifier process exit code:', verification.returncode)
if verification.returncode != 0:
    raise RuntimeError('Goldset verifier reported a failed record. Read the verifier output above.')
if 'passed 1' not in verification.stdout and 'skipped 1' not in verification.stdout:
    raise RuntimeError('Could not find a clear pass/skip summary in the verifier output.')


Goldset repository revision: fe4f9bfbcb46637481242af2315c6b59951710b4
Record: 4e210790188a0356
Fix date: 2025-01-02
Project: erikrose/parsimonious
Project license: MIT
Bug specification: Fix multiple terms as part of `/` expressions (#250)
Running one-record verification now...
verifying 1 record(s)

[1/1] pass  erikrose/parsimonious  parsimonious.grammar

passed 1   failed 0   skipped 0

Verifier messages: This clones public projects and runs their test suites on this machine. Use a container if that matters to you.


Verifier process exit code: 0


## Pilot the recommended code model

The candidate is Qwen2.5-Coder-7B-Instruct. Its full model files are about 15.2 GB, so the download may take a while. The next cells install the 4-bit loading tools, download the model into temporary Colab storage, and generate one short answer. The model files are not added to this Git repository. If Colab disconnects, rerun the setup and model cells.

In [ ]:
import subprocess
import sys

# Install only the model libraries. Keep Colab's existing PyTorch/CUDA build.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'transformers', 'accelerate', 'bitsandbytes'
], check=True)
from importlib.metadata import version
for package in ('transformers', 'accelerate', 'bitsandbytes'):
    print(f'{package}:', version(package))
print('Model libraries installed.')

transformers: 5.22.1
accelerate: 1.15.0
bitsandbytes: 0.50.2
Model libraries installed.


## Load the model and make one short request

This uses 4-bit NF4 compression to reduce GPU memory use. The T4 supports this setup with the current Colab runtime. The generated text is only a format check; it is not an experiment result.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_DIR = Path('/content/mtech-bug-fix-agent')
SOURCE_DIR = PROJECT_DIR / 'src'
if not (SOURCE_DIR / 'bugfix_agent' / 'model.py').is_file():
    raise FileNotFoundError('Run the Download and install the project cell first.')
sys.path.insert(0, str(SOURCE_DIR))
from bugfix_agent.model import HuggingFaceModel

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
torch.cuda.reset_peak_memory_stats()
model = HuggingFaceModel(MODEL_ID, max_new_tokens=128, max_input_tokens=1024)
print('Model loaded:', MODEL_ID)
print('Model snapshot:', model.revision)
print('Load time (seconds):', model.load_seconds)
print('GPU memory after load (GiB):', round(torch.cuda.memory_allocated() / (1024 ** 3), 2))

prompt = (
    'Return only a unified diff for this small Python bug. '
    'Current file: def divide(a, b): return a / b. '
    'Bug: when b is zero, return None instead of raising ZeroDivisionError.'
)
response = model.generate(prompt)
print('Generated response:')
print(response)
print('Token and time record:', model.last_usage)
print('Peak GPU memory (GiB):', round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2))
if not response.strip():
    raise RuntimeError('The model loaded but returned an empty response.')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-Coder-7B-Instruct
Model snapshot: None
Load time (seconds): 81.013
GPU memory after load (GiB): 8.34
Generated response:
```diff
diff --git a/divide.py b/divide.py
--- a/divide.py
+++ b/divide.py
@@ -1,3 +1,4 @@
 def divide(a, b):
-    return a / b
+    if b == 0:
+        return None
+    else:
+        return a / b
```
Token and time record: {'input_tokens': 70, 'output_tokens': 76, 'generation_seconds': 4.8}
Peak GPU memory (GiB): 10.49
